In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rd_seed = 42

Read the data

In [2]:
df = pd.read_csv('data.csv')

Basic info

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

Drop unnecessary columns and change diagnosis data type.

In [5]:
df = df.drop(columns=['id', 'Unnamed: 32'])
diag = {'M': 1, 'B': 0}
df['diagnosis'] = df['diagnosis'].map(diag)

Data normalization.

In [6]:
cols = df.drop('diagnosis', axis=1).columns

X = df.drop('diagnosis', axis=1).values
y = df['diagnosis'].values

standard_scaler = StandardScaler()
X_scaled = standard_scaler.fit_transform(X)

Train/val/test set split.

In [7]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X_scaled, y, test_size=0.2, random_state=rd_seed)
Xval, Xtest, yval, ytest = train_test_split(Xtest, ytest, test_size=0.5, random_state=rd_seed)

Train random forest on the train split.

In [8]:
rf = RandomForestClassifier(n_estimators=20, random_state=rd_seed)
rf.fit(Xtrain, ytrain)

RandomForestClassifier(n_estimators=20, random_state=42)

Extract rules from random forest

In [9]:
def extract_rules_from_tree(tree, column_names):
    rules = []
    tree_ = tree.tree_
    left = tree_.children_left
    right = tree_.children_right
    threshold = tree_.threshold
    features = tree_.feature
    value = tree_.value

    def recurse(node, conditions):
        if left[node] == -1:
            class_id = np.argmax(value[node][0])
            rules.append({
                'conditions': list(conditions),
                'class': class_id
            })
        else:
            feat = features[node]
            thr = threshold[node]

            conditions.append((feat, '<=', thr))
            recurse(left[node], conditions)
            conditions.pop()

            conditions.append((feat, '>', thr))
            recurse(right[node], conditions)
            conditions.pop()

    recurse(0, [])
    return rules

all_rules = []
for estimator in rf.estimators_:
    all_rules.extend(extract_rules_from_tree(estimator, cols))

We generate the RsCoverage matrix, which contains the class predicted by the rule for each pair (sample, rule from the Random Forest trees), provided the rule is applicable

In [10]:
def rule_matches_sample(rule, x_sample):
    for (feat_idx, op, thr) in rule['conditions']:
        if op == '<=':
            if not (x_sample[feat_idx] <= thr):
                return False
        else:
            if not (x_sample[feat_idx] > thr):
                return False
    return True

m = Xtrain.shape[0]
n = len(all_rules)

RsCoverage = np.full((m, n), fill_value=-1, dtype=int)
for i in range(m):
    sample = Xtrain[i]
    for j, rule in enumerate(all_rules):
        if rule_matches_sample(rule, sample):
            RsCoverage[i, j] = rule['class']
        else:
            RsCoverage[i, j] = -1

Calculate RScore for the each rule

In [11]:
def compute_RScore(RsCoverage, ytrain, k=0.00001):
    ytrain_col = ytrain[:, None]
    is_covered = RsCoverage != -1
    matches = (RsCoverage == ytrain_col)
    
    Ncc = matches.sum(axis=0)
    Nic = (is_covered & (~matches)).sum(axis=0)
    total_covered = Ncc + Nic
    
    RScore = np.full(RsCoverage.shape[1], -1.0)
    valid_rules = total_covered > 0
    
    if np.any(valid_rules):
        N_cc_v = Ncc[valid_rules]
        N_ic_v = Nic[valid_rules]
        denom_v = total_covered[valid_rules]
        RScore[valid_rules] = ((N_cc_v - N_ic_v) / denom_v) + (N_cc_v / (N_ic_v + k))
        
    return RScore

RScore = compute_RScore(RsCoverage, ytrain)

Optimizing rules set on the validation set

In [12]:
def Accuracy(rule_subset, X, y):
    major_class = np.argmax(np.bincount(y))
    correct = 0

    for x_sample, true_label in zip(X, y):
        predicted = None

        for r in rule_subset:
            rule = all_rules[r]
            if rule_matches_sample(rule, x_sample):
                predicted = rule['class']
                break

        if predicted is None:
            predicted = major_class

        if predicted == true_label:
            correct += 1

    return correct / len(y)


def getRuleSet(RScore, n, iniRuleNo):
    scores = RScore - RScore.min() + 1e-5
    probs = scores / scores.sum()
    chosen = np.random.choice(n, size=iniRuleNo, replace=False, p=probs)
    return chosen.tolist()


def heuristSearch(current_set, RScore):
    current_accuracy = Accuracy(current_set, Xval, yval)
    new_set = current_set.copy()

    if np.random.rand() < 0.5 and len(new_set) > 1:
        idx_to_remove = np.random.choice(new_set)
        candidate = [r for r in new_set if r != idx_to_remove]

        if Accuracy(candidate, Xval, yval) > current_accuracy:
            return candidate

    else:
        candidates = np.setdiff1d(np.arange(len(all_rules)), new_set)
        if len(candidates) > 0:
            idx_to_add = np.random.choice(candidates)
            candidate = new_set + [idx_to_add]

            if Accuracy(candidate, Xval, yval) > current_accuracy:
                return candidate

    return current_set


In [13]:
MaxIteration = 100
rulenum = 20

iniRs = getRuleSet(RScore, n, rulenum)
impRs = iniRs.copy()
bestRs = iniRs.copy()
bestAcc = Accuracy(bestRs, Xval, yval)

for i in range(MaxIteration):
    impRs = heuristSearch(impRs, RScore)
    impAcc = Accuracy(impRs, Xval, yval)

    if impAcc > bestAcc:
        bestRs = impRs.copy()
        bestAcc = impAcc

    if i % 10 == 0:
        impRs = getRuleSet(RScore, n, rulenum)

Check the accuracy on test set.

In [14]:
test_accuracy = Accuracy(bestRs, Xtest, ytest)
print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.9825


Print best rules

In [15]:
best_rules = [all_rules[r] for r in bestRs]


for i, rule in enumerate(best_rules, start=1):
    conditions = rule['conditions']
    class_id = rule['class']

    condition_str = []
    for (feat_idx, op, thr) in conditions:
        condition_str.append(f"{cols[feat_idx]} {op} {thr:.4f}")

    if_str = " AND ".join(condition_str)

    print(f"Rule {i}: IF {if_str} THEN class = {class_id}\n")

Rule 1: IF area_worst <= -0.0101 AND concave points_worst <= 0.7102 AND texture_mean <= 0.4585 AND fractal_dimension_worst <= -0.8105 AND concave points_mean <= -0.4374 THEN class = 0

Rule 2: IF radius_se <= 0.0551 AND concave points_worst > 0.6668 AND concavity_worst <= 4.5040 THEN class = 1

Rule 3: IF area_mean > 0.1117 AND concave points_worst > 0.4841 THEN class = 1

Rule 4: IF compactness_mean <= -0.0453 AND concavity_worst <= -0.2676 AND concave points_worst > -0.0770 AND texture_worst > -0.4001 AND concavity_worst <= -0.4097 THEN class = 0

Rule 5: IF radius_mean <= 0.2550 AND concave points_worst > 0.4171 AND area_worst > -0.6827 AND symmetry_worst > -0.2358 AND fractal_dimension_se > -0.3570 THEN class = 1

Rule 6: IF area_worst > -0.0101 AND texture_worst > -0.9392 AND concave points_mean > 0.0301 THEN class = 1

Rule 7: IF compactness_worst <= 0.1100 AND concave points_worst > 0.3227 AND perimeter_se > -0.4171 THEN class = 1

Rule 8: IF concave points_worst > 0.4773 AND co